# Sprint 2 · Ingeniería de atributos (features)

**Tesis:** Diagnóstico de causalidad de fallas en una red telco usando grafos · Región **Pasco**, Perú.

El motor no elige entre alarmas sueltas sino entre **candidatos**: cada equipo con alarmas dentro de un incidente. Este
notebook construye la tabla de candidatos con cuatro familias de atributos (`src/features.py`, diccionario en
`docs/05_features.md`):

| Familia | Qué captura |
|---|---|
| Temporal | cuándo y cuánto alarmó el equipo dentro del incidente |
| Topológica | dónde está en la red y cuántos equipos alarmados tiene aguas abajo |
| Co-ocurrencia | qué tipo de alarma suele llegar primero / preceder a otro (**aprendido solo en entrenamiento**) |
| Semántica | rol del equipo y tipo de alarma según el catálogo del manual |

Las reglas de los manuales **no** son features: son la etiqueta plata (usarlas como entrada sería circular).

**Contenido** 1. Datos · 2. Diccionario · 3. Tabla de candidatos por fold (sin fuga) · 4. Distribuciones ·
5. ¿Separan los atributos a la causa raíz? (datos simulados) · 6. Exportación · 7. Conclusiones


## 1. Datos, grafo e incidentes

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.ingesta import construir_dataset
from src.grafo import construir_grafo, grafo_sitios, jerarquia_sitios
from src.etiquetado import construir_incidentes
from src.evaluacion import folds_temporales, verdad_desde_simulacion
from src.features import ExtractorFeatures, diccionario_features
from src.sintetico import simular_avalanchas

BASE_DIR = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
CATALOGO = ROOT / "docs" / "catalogo_alarmas.csv"

df = pd.read_parquet(PROC / "dataset_consolidado.parquet") if (PROC / "dataset_consolidado.parquet").exists() \
    else construir_dataset(str(BASE_DIR))
G = construir_grafo(str(BASE_DIR / "red.xlsx"), verbose=False)
J = jerarquia_sitios(G, grafo_sitios(G))
catalogo = pd.read_csv(CATALOGO) if CATALOGO.exists() else None
df_inc, incidentes = construir_incidentes(df, G)
print(f"{len(incidentes):,} incidentes · {len(df_inc):,} alarmas en incidentes · "
      f"catálogo: {0 if catalogo is None else len(catalogo)} alarmas de manual")

> **Resumen.** Reutiliza el dataset del Sprint 1, el grafo del notebook 04 y los incidentes del notebook 05 (alarmas nuevas conectadas a ≤ 4 saltos). El catálogo de alarmas viene del manual del RTN 320.

## 2. Diccionario de atributos

In [ ]:
display(diccionario_features())

> **Qué muestra la tabla.** Cada atributo con su familia y definición. Es la versión resumida de `docs/05_features.md`.

## 3. Tabla de candidatos por fold (sin fuga de información)

Los atributos de **co-ocurrencia** se aprenden con `fit` usando SOLO las ventanas de entrenamiento de cada fold y se
aplican con `transform` a las de prueba. Así ningún atributo «ve» el futuro.

In [ ]:
folds = folds_temporales(df[["ventana_id", "ventana_ts"]], n_folds=4, min_entrenamiento=0.4, separacion=1,
                        pesos=incidentes.groupby("ventana_id").size())
tablas = []
for f in folds:
    tr = df_inc[df_inc["ventana_id"].isin(f["train"])]
    te = df_inc[df_inc["ventana_id"].isin(f["test"])]
    if tr.empty or te.empty:
        continue
    fx = ExtractorFeatures(G, catalogo=catalogo, J=J).fit(tr)
    tablas.append(fx.transform(te).assign(fold=f["fold"]))
X = pd.concat(tablas, ignore_index=True) if tablas else pd.DataFrame()
print(f"Candidatos: {len(X):,} en {X['incidente_id'].nunique() if len(X) else 0:,} incidentes de prueba")
display(X.head(8) if len(X) else X)

> **Qué muestra la tabla.** Una fila por candidato (equipo con alarmas en un incidente de prueba) con sus atributos. `fold` indica en qué bloque de prueba cayó; los atributos de co-ocurrencia de esa fila salen solo del pasado.

## 4. Distribuciones

In [ ]:
num = ["t_rel_s", "pct_orden", "sev_max", "saltos_rel", "pct_alarmados_abajo", "dist_media_alarmados",
       "p_primero_tipo", "p_precede_media"]
if len(X):
    fig, axs = plt.subplots(2, 4, figsize=(13, 5.2))
    for ax, col in zip(axs.flat, num):
        X[col].dropna().plot.hist(bins=20, ax=ax, color="#0B5D8F")
        ax.set_title(col, fontsize=10); ax.set_ylabel("")
    plt.suptitle("Distribución de los atributos (red real, incidentes de prueba)"); plt.tight_layout(); plt.show()
else:
    print("Sin candidatos con los datos actuales.")

> **Qué muestran los gráficos.** La forma de cada atributo en la red real. Sirve para detectar valores raros (p. ej. `t_rel_s` muy grandes por alarmas que llegan tarde) y para decidir normalizaciones antes del motor.

In [ ]:
if len(X):
    display(X["rol"].value_counts().rename("candidatos").to_frame().T)
    display(X["tipo_alarma_manual"].value_counts().rename("candidatos").to_frame().T)

> **Qué muestran las tablas.** Cuántos candidatos hay de cada rol de equipo y de cada tipo de alarma del manual. `sin_catalogo` = alarmas de routers, que esperan los manuales del ATN 910C / NE40E.

## 5. ¿Separan los atributos a la causa raíz? (datos simulados)

En la red real todavía no hay etiquetas. Con avalanchas simuladas sobre la topología real (causa conocida) se puede ver
qué atributos distinguen a la causa raíz del resto. **Es una verificación del método, no un resultado de la red.**

In [ ]:
df_sim, verdad_sim, _ = simular_avalanchas(G, n_ventanas=500, semilla=42)
inc_df_sim, inc_sim = construir_incidentes(df_sim, G)
V = verdad_desde_simulacion(inc_df_sim, verdad_sim)
f_sim = folds_temporales(inc_df_sim[["ventana_id", "ventana_ts"]], n_folds=4)
Xs = []
for f in f_sim:
    tr = inc_df_sim[inc_df_sim["ventana_id"].isin(f["train"])]
    te = inc_df_sim[inc_df_sim["ventana_id"].isin(f["test"])]
    Xs.append(ExtractorFeatures(G, J=J).fit(tr).transform(te))
Xs = pd.concat(Xs, ignore_index=True).merge(V, on="incidente_id")
Xs["es_causa"] = Xs["equipo"] == Xs["causa_raiz_equipo"]
sep = Xs.groupby("es_causa")[num].mean().T.rename(columns={False: "otros", True: "causa raíz"})
sep["diferencia"] = sep["causa raíz"] - sep["otros"]
display(sep.round(3))

> **Qué muestra la tabla.** El promedio de cada atributo para la causa raíz y para el resto de candidatos. Las diferencias grandes indican atributos útiles: la causa suele llegar antes (`pct_orden` menor), estar más aguas arriba (`saltos_rel` menor) y tener más equipos alarmados aguas abajo (`pct_alarmados_abajo` mayor). Ojo: el simulador propaga las fallas solo aguas abajo, así que los atributos topológicos se ven más fuertes aquí de lo que serán en la red real.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(13, 3))
for ax, col in zip(axs, ["pct_orden", "saltos_rel", "pct_alarmados_abajo", "p_primero_tipo"]):
    for v, color, lab in [(False, "#9FB0C0", "otros"), (True, "#D1495B", "causa raíz")]:
        Xs.loc[Xs["es_causa"] == v, col].plot.hist(bins=15, ax=ax, alpha=0.7, color=color, density=True, label=lab)
    ax.set_title(col, fontsize=10); ax.set_ylabel("")
axs[0].legend(frameon=False); plt.suptitle("Causa raíz vs. resto (simulación)"); plt.tight_layout(); plt.show()

> **Qué muestran los gráficos.** Las distribuciones de cuatro atributos para la causa raíz (rojo) y el resto (gris). Cuanto menos se solapan, más ayuda el atributo a encontrar la causa.

## 6. Exportación

In [ ]:
if len(X):
    X.to_parquet(PROC / "features_candidatos.parquet", index=False)
    print("Guardado:", PROC / "features_candidatos.parquet")

> **Resumen.** La tabla queda en `data/processed/` (no se versiona: se regenera con este notebook).

## 7. Conclusiones

- La unidad de decisión es el **candidato** (equipo con alarmas en un incidente); hay 20 atributos en 4 familias.
- Los atributos de co-ocurrencia se ajustan por fold → no hay fuga del futuro.
- En simulación, los atributos de orden temporal, posición aguas arriba y alcance aguas abajo separan a la causa raíz.
- El motor v1 (notebook 07) usa la misma información temporal y topológica dentro de un modelo de Hawkes; esta tabla
  queda lista para un motor supervisado (Sprint 3) cuando existan suficientes etiquetas oro.